# W-BOT v4 — Qwen3-4B Fine-Tune (Colab A100)

**Dataset:** wbot_v4_train.jsonl (3605 kayıt)
**Base model:** Qwen/Qwen3-4B
**Platform:** Google Colab A100-SXM4-40GB
**Çıktı:** Qwen3-4B-wbot_v4-Q4_K_M.gguf

wbot_v3'ten fark: +605 yeni kayıt (A ve B paketleri)
  - E19/W15 fix (açıklama → Getireyim mi?)
  - W16 fix (alerji + öneri kombinasyonu)
  - S12 koşulsuz sipariş özeti
  - S29/S30/S32 kötü niyet senaryoları
  + diğer yeni senaryolar (belirsiz girdi, modifikasyon, pratik soru vb.)

Model, hiperparametreler ve GGUF dönüşüm adımları wbot_v3 ile aynı —
tek fark dataset. Eğitim scripti (`train_wbot_v2.py`) repodan olduğu
gibi çağrılır, notebook içine kopyalanmaz.

## 2. GPU Kontrolü

Runtime > Change runtime type > A100 GPU seçili olmalı.

In [ ]:
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
assert torch.cuda.is_available(), "GPU bulunamadı!"

## 3. Bağımlılık Kurulumu

`robot_waiter_ai/training/requirements_train.txt` ile birebir aynı paket ve
versiyon pinleri (repo henüz klonlanmadığı için burada elle yazıldı —
transformers>=4.43.0 Qwen3 chat template desteği için zorunlu).

In [ ]:
!pip install -q "transformers>=4.43.0" "datasets>=2.20.0" "bitsandbytes>=0.46.1" \
    "peft>=0.10.0" "accelerate>=0.27.0"

## 4. Google Drive Mount

`/content/drive/MyDrive/wbot/` altında dataset ve çıktıların bulunması
beklenir. Kendi Drive yapınıza göre aşağıdaki path'leri güncelleyin.
`wbot_v4_train.jsonl` henüz repoya commit edilmediği için buraya elle
yüklemeniz gerekiyor.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 5. Dataset Hazırlığı

In [ ]:
# Dataset path — kendi Drive klasörünüze göre güncelleyin
DATASET_PATH = "/content/drive/MyDrive/wbot/wbot_v4_train.jsonl"
OUTPUT_DIR   = "/content/drive/MyDrive/wbot/wbot_v4_output"
MODEL_NAME   = "Qwen/Qwen3-4B"

# Kontrol
import os
assert os.path.exists(DATASET_PATH), f"Dataset bulunamadı: {DATASET_PATH}"
import subprocess
line_count = int(subprocess.check_output(["wc", "-l", DATASET_PATH]).split()[0])
print(f"Dataset: {line_count} kayıt (beklenen: 3605)")
assert line_count == 3605, f"Beklenmeyen kayıt sayısı: {line_count}"

## 6. Eğitim

`train_wbot_v2.py` repodan klonlanıp olduğu gibi çağrılır (script notebook
içine kopyalanmaz). Gerçek CLI argümanları (`train_wbot_v2.py` argparse
tanımından — tire ile, alt çizgi değil): `--dataset`, `--output-dir`,
`--epochs`, `--run-eval`.

**Repo private ise:** Colab sol panelden 🔑 **Secrets** sekmesine gidip
`GITHUB_TOKEN` adında bir secret ekleyin (GitHub → Settings → Developer
settings → Personal access tokens → en az `repo` yetkili bir token
oluşturup buraya yapıştırın). Public repo ise bu adım gerekmez, token
bulunamazsa otomatik olarak token'sız clone denenir.

In [ ]:
import os

REPO_DIR = "/content/Garson-bot"
REPO_URL = "https://github.com/MustafaEmreBiyik/Garson-bot.git"

# Repo private ise Colab Secrets'tan GITHUB_TOKEN okunur (yoksa token'sız denenir)
try:
    from google.colab import userdata
    _token = userdata.get("GITHUB_TOKEN")
except Exception:
    _token = None

clone_url = REPO_URL if not _token else REPO_URL.replace("https://", f"https://{_token}@")

if not os.path.exists(REPO_DIR):
    !git clone {clone_url} {REPO_DIR}
    # Alternatif — repoyu elle Drive'a yükleyip buradan kopyalamak isterseniz:
    # !cp -r /content/drive/MyDrive/wbot/Garson-bot {REPO_DIR}
else:
    print("Repo zaten mevcut:", REPO_DIR)

%cd {REPO_DIR}

In [ ]:
# --drive-dir kullanılmadı: OUTPUT_DIR zaten Drive path'inde (Hücre 5),
# checkpoint'ler eğitim sırasında doğrudan Drive'a yazılır.
!python robot_waiter_ai/training/train_wbot_v2.py \
    --dataset "{DATASET_PATH}" \
    --output-dir "{OUTPUT_DIR}" \
    --epochs 3 \
    --run-eval

## 7. GGUF Dönüşümü

PROJE_DURUMU.md → "GGUF Dönüşümü — Colab Hücreleri" ile birebir aynı akış,
dosya adları wbot_v4 olacak şekilde güncellendi.

In [ ]:
# Kurulum
!pip install -q transformers peft accelerate
!apt-get install -q -y build-essential cmake
!git clone https://github.com/ggml-org/llama.cpp /content/llama.cpp
!cd /content/llama.cpp && cmake -B build -DGGML_CUDA=ON && cmake --build build --config Release -j4

In [ ]:
ADAPTER_DIR = f"{OUTPUT_DIR}/adapter"
MERGED_DIR  = "/content/wbot_v4_merged"
GGUF_PATH   = "/content/drive/MyDrive/wbot/Qwen3-4B-wbot_v4-Q4_K_M.gguf"

In [ ]:
# Merge (adapter + base model)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

print("Base model yükleniyor...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16, device_map="cpu"
)
model = PeftModel.from_pretrained(model, ADAPTER_DIR)
print("Merge ediliyor...")
model = model.merge_and_unload()
model.save_pretrained(MERGED_DIR)
AutoTokenizer.from_pretrained(ADAPTER_DIR).save_pretrained(MERGED_DIR)
print("Merge tamam:", MERGED_DIR)

In [ ]:
# GGUF dönüşümü
!python /content/llama.cpp/convert_hf_to_gguf.py {MERGED_DIR} \
    --outtype q4_k_m \
    --outfile {GGUF_PATH}
print("GGUF kaydedildi:", GGUF_PATH)

## 8. Drive'a Kaydet

`GGUF_PATH` (Hücre 7) zaten doğrudan Drive'a yazıldı — bu hücre yalnızca
dosyanın var olduğunu doğrular. `GGUF_PATH` lokal (`/content/...`) bir yola
ayarlanmışsa aşağıdaki yorum satırını açarak Drive'a kopyalayın.

In [ ]:
import os
import shutil

GGUF_OUTPUT = "/content/drive/MyDrive/wbot/Qwen3-4B-wbot_v4-Q4_K_M.gguf"

# GGUF_PATH lokal bir yoldaysa (Drive değilse) önce buraya kopyalayın:
# shutil.copy(GGUF_PATH, GGUF_OUTPUT)

assert os.path.exists(GGUF_OUTPUT), f"GGUF bulunamadı: {GGUF_OUTPUT}"
size_mb = os.path.getsize(GGUF_OUTPUT) / 1e6
print(f"✓ GGUF kaydedildi: {GGUF_OUTPUT} ({size_mb:.0f} MB)")

## Eğitim Tamamlandı

Sonraki adımlar:
1. GGUF dosyasını Jetson'a kopyala: `/home/emk/models/`
2. `demo_usb.py`'deki model path'i güncelle
3. Eval: `python3 scripts/eval_gguf.py` → hedef 32/32
4. V01-V06: `python3 scripts/eval_gguf.py --v4-targets`